# PyTorch: tensors, autograd, and neural nets that run as you write them

_PyTorch (a complete course)_

**A Python library where every operation runs immediately (eager / define-by-run), so you build, debug, and train models with ordinary Python — which is why research lives here.**

PyTorch is three ideas wearing one coat. Strip it down and a deep-learning framework is just:

       
         
- Tensors &mdash; a fast, n-dimensional number array (think NumPy) that can also run on a
         GPU and remember the operations performed on it.
         
- Automatic differentiation (autograd) &mdash; the machinery that, given any computation you wrote,
         computes the gradient of the result with respect to every input. This is dl-backprop done for
         you, automatically, for arbitrary code.
         
- Neural-network building blocks (torch.nn) &mdash; ready-made layers, activations,
         and losses you snap together into a model, plus optimizers (torch.optim) that apply the
         gradient updates.
       

---

This notebook is a practice scaffold for the **PyTorch: tensors, autograd, and neural nets that run as you write them** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
# Colab note: torch is preinstalled — no pip install needed. Just run this cell.
import torch
import torch.nn as nn

# ============================================================
# 1. VERSION + HARDWARE CHECK
#    torch.cuda.is_available() -> True if a GPU is present and usable.
#    Pick ONE device string and use it everywhere (avoids CPU/GPU mismatch).
# ============================================================
print("PyTorch version:", torch.__version__)        # e.g. 2.x.x
print("CUDA available :", torch.cuda.is_available()) # True on a GPU runtime
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device   :", device)
if device == "cuda":
    print("GPU name       :", torch.cuda.get_device_name(0))

# Set a seed so results are reproducible (random init, shuffling, dropout all use RNG).
torch.manual_seed(0)

# ============================================================
# 2. MAKE A COUPLE OF TENSORS AND DO AN OP
#    Each line runs IMMEDIATELY (eager / define-by-run) — no graph to compile,
#    no session to open. You can print any intermediate value on the spot.
# ============================================================
a = torch.tensor([1.0, 2.0, 3.0])   # a 1-D tensor (like a NumPy array)
b = torch.ones(3)                    # tensor([1., 1., 1.])
c = a + b                            # runs NOW -> tensor([2., 3., 4.])
print("a + b =", c)
print("shape :", c.shape, "| dtype:", c.dtype)

# Tensors interoperate with NumPy, but remember: a tensor can carry an autograd
# history AND lives on a specific device — two things a NumPy array does not.
print("as numpy:", c.numpy())

# ============================================================
# 3. MOVE TO THE GPU IF ONE IS AVAILABLE
#    .to(device) places the tensor on the chosen device.
# ============================================================
a = a.to(device)
print("a is now on:", a.device)      # cpu, or cuda:0 on a GPU runtime

# ============================================================
# 4. A FIVE-LINE LINEAR MODEL — FORWARD PASS (a teaser)
#    nn.Linear(3, 1) computes y = W x + b. Move the model to the SAME device.
#    This is the whole "model" step of the standard workflow, in one layer.
# ============================================================
model = nn.Linear(in_features=3, out_features=1).to(device)
x = torch.tensor([1.0, 2.0, 3.0], device=device)   # one input on the right device
y = model(x)                                        # forward pass — runs immediately
print("model(x) =", y.item())                       # a single predicted number

# That's the skeleton: tensors -> model -> (add a loss + optimizer + loop) -> train.
# The rest of the course unpacks each step.

## Visualize the data & results

_What does 'training' actually look like? A tiny model's loss falling over epochs — the curve you watch in every PyTorch run. Here: full-batch gradient descent fitting a line y = 2x + 1, the same loop PyTorch automates with autograd + an optimizer._

In [ ]:
import numpy as np

# ---- A tiny training loop done by hand (this is what PyTorch automates) ----
np.random.seed(0)
N = 64
x = np.linspace(-1, 1, N)
y_true = 2.0 * x + 1.0 + 0.15 * np.random.randn(N)   # line y = 2x + 1 + noise

w, b = 0.0, 0.0       # parameters PyTorch would store in nn.Linear
lr = 0.3              # learning rate (optimizer step size)
epochs = 40
losses = []
for ep in range(epochs):
    yhat = w * x + b                 # FORWARD pass  -> model(x)
    err  = yhat - y_true
    loss = np.mean(err ** 2)         # MSE LOSS      -> loss = criterion(yhat, y)
    losses.append(round(float(loss), 4))
    # BACKWARD: hand-derived gradients. In PyTorch, loss.backward() does this
    # automatically via autograd — no calculus by hand.
    gw = 2 * np.mean(err * x)
    gb = 2 * np.mean(err)
    # STEP: gradient-descent update -> optimizer.step()
    w -= lr * gw
    b -= lr * gb

print("recovered w, b:", round(w, 3), round(b, 3))   # -> 1.889 1.002
print("loss[0], loss[-1]:", losses[0], losses[-1])    # -> 2.2537 0.0211
points = [[ep, l] for ep, l in enumerate(losses)]
print(points)   # the (epoch, loss) pairs plotted above

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate coming from TensorFlow 1 asks why, in PyTorch, they can put a plain Python print(x) in the middle of their model's forward method and actually see numbers. What is the one-word property that makes this work, and what does it mean?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how PyTorch executes a model's operations. — _PyTorch runs each operation the moment Python reaches it, rather than building a graph to run later._
- Name the property: eager execution (define-by-run). — _Because the graph is built as the code runs, real numbers exist at every line, so a normal print sees them._
- Contrast with old static-graph TensorFlow 1. — _There the graph was defined first and run later in a session, so at definition time no numbers existed to print._

**Answer:** The property is eager execution (also called define-by-run): every operation runs immediately as Python reaches it, building the computation graph on the fly. Because real values exist at every line, an ordinary print, breakpoint, or if works inside the model. Old static-graph TensorFlow 1 defined the whole graph first and only produced numbers later in a separate session, which is why you could not just print mid-network. This define-by-run feel is the main reason PyTorch dominates research.

</details>

**Problem 2.** You write model = model.to("cuda") at the top of your script, but the first batch crashes with Expected all tensors to be on the same device, but found at least two devices, cuda and cpu. What did you forget, and what is the general rule?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the error: two devices are involved, cuda and cpu. — _An operation is mixing a tensor on the GPU with a tensor on the CPU, which PyTorch forbids._
- Check where the input batch lives. — _You moved the model to the GPU but a freshly loaded input tensor defaults to the CPU, so they mismatch._
- Move the inputs to the same device as the model. — _x = x.to(device) (and the targets too) puts every tensor on one device so the operation is legal._

**Answer:** You moved the model to the GPU but left the input batch on the CPU. New tensors default to the CPU, so the forward pass tried to combine a GPU weight with a CPU input. The general rule: pick one device up front and send everything there &mdash; model.to(device) and x = x.to(device) (and the targets). Mixing CPU and GPU tensors is one of the most common PyTorch errors.

</details>

**Problem 3.** Your friend wants to fit a standard, off-the-shelf image classifier as fast as possible with minimal code, while you are inventing a brand-new attention variant for a paper. Which tools fit each of you, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Characterize the friend's task: a standard architecture, wants minimal boilerplate. — _A high-level wrapper hides the training loop and gives a short, batteries-included path for common models._
- Characterize your task: a novel custom component for research. — _Research needs full control and easy debugging of every tensor, which raw PyTorch's eager style gives._
- Match each person to a tool. — _Friend -> Keras / fast.ai / PyTorch Lightning; you -> raw PyTorch (and Lightning still sits on PyTorch if wanted)._

**Answer:** Your friend's job is a standard model with minimal code, so a high-level wrapper &mdash; Keras, fast.ai, or PyTorch Lightning &mdash; is the fast path; it hides the loop and the boilerplate. Your job is a novel custom component for a paper, where you want to see and touch every tensor and debug freely, so raw PyTorch with its eager, define-by-run style fits best. Note these are not opposites: Lightning runs on top of PyTorch, so you can always start in plain PyTorch and add a wrapper later. (JAX would be the pick only if you specifically wanted a functional style or heavy TPU use.)

</details>